# R4 practice: data you can defend

**Starter notebook for research studio R4** of *Energy Systems, Step by Step*, by Honglin Li. About 2 hours with the
practice tasks at the end.

You will clean the same file as the studio page, with the same rules, and write the same log: SPP's 2025 hourly demand,
wind and solar from EIA's Form EIA-930 (`b02-python-data.json`, an EIA-adjusted extract: each fuel's adjusted
generation columns summed and negative values set to 0; nothing else repaired). Every step ends with an **auto-check
cell**: an `assert` that stays silent when the result matches the page.

**How to run it**

- *Google Colab:* open a new notebook, choose *File → Upload notebook*, then upload `b02-python-data.json` with the
  folder icon on the left.
- *Locally:* Python 3 with pandas and NumPy; keep the data file next to this notebook.

Download the data file from the studio page (Sources: "Raw SPP 2025 file").

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

assert Path("b02-python-data.json").exists(), "b02-python-data.json not found: put it next to this notebook (in Colab, upload it with the folder icon)."
with open("b02-python-data.json") as f:
    raw = json.load(f)
meta = raw["meta"]
ends = pd.date_range(raw["start_utc"], periods=len(raw["demand"]), freq="h", tz="UTC")   # EIA-930: each stamp ENDS its hour, in UTC
starts = ends - pd.Timedelta(hours=1)                                                    # label each hour by its start
data = {c: np.array(raw[c], dtype=float) for c in ["demand", "wind", "solar"]}
print(meta["title"])
print("Extraction:", meta["not_cleaned"])
print(len(ends), "rows; first hour starts", starts[0], "and the last ends", ends[-1])

In [ ]:
# auto-check
assert len(ends) == 8760 and all(len(v) == 8760 for v in data.values())
print("Check passed: 8,760 hours, three columns, hour-ending UTC stamps.")

## Part 1 · The page's rules, as functions

The same rules as the studio page, so your numbers can be compared with it:

- **Spike rule** (demand only): `half` flags demand under half the year's median; `sd3` flags demand more than 3
  standard deviations below the mean; `keep` flags nothing. A flag is provisional: check it against the operator's
  records before you call it an error.
- **Frozen-reading rule:** runs of `min_run` or more identical **non-zero** readings are suspected stuck feeds
  (zeros are fine: solar is zero every night). `0` turns the rule off.
- **Fill:** `same` uses the mean of the **same UTC hour** (rows 24 apart) on valid days within 7, then 21, then 45 days
  each side; near the daylight-saving changes that is one local hour off. `linear` draws a straight line across each
  gap; `drop` leaves the flagged hours out of every total.
- **Time base:** `utc` reports in UTC; `label` keeps the UTC rows and labels results in Central time with daylight
  saving; `cst` labels in UTC−6 all year; `force` puts the rows on the local clock and forces every day to 24 rows.

In [ ]:
WINDOWS = [7, 21, 45]   # days each side for the same-hour fill
MONTHS = ["January", "February", "March", "April", "May", "June", "July", "August", "September", "October", "November", "December"]

def spike_flags(a, rule):
    if rule == "keep":
        return np.zeros(len(a), bool)
    limit = a.mean() - 3 * a.std(ddof=1) if rule == "sd3" else 0.5 * np.median(a)
    return a < limit

def run_flags(a, min_run):
    f = np.zeros(len(a), bool)
    if not min_run:
        return f
    i = 0
    while i < len(a):
        j = i
        while j + 1 < len(a) and a[j + 1] == a[i]:
            j += 1
        if a[i] != 0 and j - i + 1 >= min_run:
            f[i:j + 1] = True
        i = j + 1
    return f

def fill(a, flags, method):
    out, n = a.copy(), len(a)
    if method == "drop":
        out[flags] = np.nan
        return out
    if method == "linear":
        i = 0
        while i < n:
            if not flags[i]:
                i += 1
                continue
            j = i
            while j < n and flags[j]:
                j += 1
            left = a[i - 1] if i > 0 else a[j]
            right = a[j] if j < n else left
            for k in range(i, j):
                out[k] = left + (right - left) * (k - i + 1) / (j - i + 1)
            i = j
        return out
    for i in np.flatnonzero(flags):                 # "same"
        out[i] = 0.0
        for w in WINDOWS:
            q = i + 24 * np.arange(-w, w + 1)
            q = q[(q != i) & (q >= 0) & (q < n)]
            q = q[~flags[q]]
            if len(q):
                out[i] = a[q].mean()
                break
    return out

In [ ]:
DEFAULT = {"time": "label", "spike": "half", "frozen": 3, "fill": "same"}   # the course's choices
RAW = {"time": "utc", "spike": "keep", "frozen": 0, "fill": "same"}         # no rules at all

def force_rows(n):
    # the original row each output row holds when every local day is forced to 24 rows: the hour starting 01:00 CST on
    # 9 March is copied where the clock jumps, and the second 01:00 hour of 2 November is dropped
    add = int(np.flatnonzero(starts == pd.Timestamp("2025-03-09 08:00", tz="UTC"))[0])
    drop = int(np.flatnonzero(starts == pd.Timestamp("2025-11-02 07:00", tz="UTC"))[0])
    return np.r_[np.arange(0, add), add - 1, np.arange(add, drop), np.arange(drop + 1, n)]

def clean(ch):
    ch = {**DEFAULT, **ch}
    out, flags, counts = {}, {}, {}
    for c in ["demand", "wind", "solar"]:
        f = run_flags(data[c], ch["frozen"])
        sp = spike_flags(data[c], ch["spike"]) if c == "demand" else np.zeros(len(f), bool)
        counts[c] = {"frozen": int(f.sum()), "spike": int((sp & ~f).sum())}
        flags[c] = f | sp
        out[c] = fill(data[c], flags[c], ch["fill"])
    rows = force_rows(len(ends)) if ch["time"] == "force" else np.arange(len(ends))
    return {"choice": ch, "cols": {c: v[rows] for c, v in out.items()}, "flags": {c: v[rows] for c, v in flags.items()}, "rows": rows, "counts": counts}

def hour_label(row, base):
    t = starts[row]
    if base == "utc":
        zone = "UTC"
    elif base == "cst":
        t, zone = t + pd.Timedelta(hours=-6), "CST"
    else:
        t = t.tz_convert("America/Chicago")
        zone = t.tzname()
    return f"{t.hour:02d}:00–{(t.hour + 1) % 24:02d}:00 {zone}, {t.day} {MONTHS[t.month - 1]}"

def report(R):
    d, ch = R["cols"]["demand"], R["choice"]
    base = ch["time"] if ch["time"] in ("utc", "cst") else "local"
    lo, hi = int(np.nanargmin(d)), int(np.nanargmax(d))
    return {"demand TWh": np.nansum(d) / 1e6, "lowest hour GW": d[lo] / 1e3,
            "peak hour": hour_label(R["rows"][hi], base), "wind TWh": np.nansum(R["cols"]["wind"]) / 1e6}

def show(rep):
    return {"demand TWh": f"{rep['demand TWh']:.2f}", "lowest hour GW": f"{rep['lowest hour GW']:.1f}",
            "peak hour": rep["peak hour"], "wind TWh": f"{rep['wind TWh']:.2f}"}

print("raw:      ", show(report(clean(RAW))))
print("course's: ", show(report(clean(DEFAULT))))

In [ ]:
# auto-check: the page's four numbers, raw and with the course's choices
assert show(report(clean(RAW))) == {"demand TWh": "299.48", "lowest hour GW": "1.5", "peak hour": "21:00–22:00 UTC, 29 July", "wind TWh": "110.46"}
assert show(report(clean(DEFAULT))) == {"demand TWh": "298.56", "lowest hour GW": "24.4", "peak hour": "16:00–17:00 CDT, 29 July", "wind TWh": "110.73"}
c = clean(DEFAULT)["counts"]
assert (c["demand"]["frozen"] + c["demand"]["spike"], c["wind"]["frozen"], c["solar"]["frozen"]) == (699, 671, 1377)
print("Check passed: the same four numbers as the page, and the course's data-build counts (699, 671, 1,377).")

## Part 2 · Your choices, and what each one changes

### Step 1 · Your four numbers

Set `MINE` to the choices you made on the page and compare each number with its raw value **in percent**. The peak's
clock time is a label, not a quantity: compare it separately.

In [ ]:
MINE = {"time": "label", "spike": "half", "frozen": 3, "fill": "same"}   # edit these

raw_rep, my_rep = report(clean(RAW)), report(clean(MINE))
for k in ["demand TWh", "lowest hour GW", "wind TWh"]:
    print(f"{k:15s} {my_rep[k]:10.2f}   raw {raw_rep[k]:10.2f}   change {100 * (my_rep[k] - raw_rep[k]) / raw_rep[k]:+7.1f}%")
print("peak hour      ", my_rep["peak hour"], "  raw", raw_rep["peak hour"])

### Step 2 · One choice at a time

Vary one decision at a time from your choices and keep the others. This table is what defends your choices: it shows
which numbers each one moves.

In [ ]:
OPTIONS = {"time": ["utc", "label", "cst", "force"], "spike": ["keep", "half", "sd3"],
           "frozen": [0, 2, 3, 4, 6, 12, 24], "fill": ["drop", "linear", "same"]}

def sensitivity(ch):
    rows = [{"change": "your choices", **show(report(clean(ch)))}]
    for key, opts in OPTIONS.items():
        for o in opts:
            if o != ch[key]:
                rows.append({"change": f"{key} = {o}", **show(report(clean({**ch, key: o})))})
    return pd.DataFrame(rows).set_index("change")

table = sensitivity(MINE)
table

In [ ]:
# auto-check (with the course's choices)
t = sensitivity(DEFAULT)
assert len(t) == 1 + 13 and t.loc["fill = drop", "demand TWh"] == "275.09"
print("Check passed: 13 alternatives; leaving the flagged hours out gives 275.09 TWh, 23.47 TWh less than the same-hour fill.")

### Step 3 · A check the page's rules do not pass

Cleaning is not finished when the rules have run: check the result. Between midnight and 4 a.m. local time solar
output must be near zero (lesson B2's physics check: above 5% of the year's peak fails).

In [ ]:
def night_solar(R):
    rows = R["rows"]
    local_hour = starts[rows].tz_convert("America/Chicago").hour
    sol = R["cols"]["solar"]
    bad = (local_hour < 4) & (sol > 0.05 * np.nanmax(sol))
    return int(bad.sum()), int((bad & R["flags"]["solar"]).sum())

for f in ["same", "linear", "drop"]:
    n, filled = night_solar(clean({**DEFAULT, "fill": f}))
    print(f"fill = {f:6s}: {n:3d} night hours with solar above 5% of its peak, {filled} of them filled")

In [ ]:
# auto-check
assert night_solar(clean(DEFAULT)) == (65, 49)
print("Check passed: with the course's choices 65 night hours fail, 49 of them filled by the same-hour rule. Log it as a known limit, then fix it (practice 2).")

### Step 4 · The data note

The note is the audit trail: where the data came from, what was done, how many hours each rule touched, and what each
choice changed. Fill in your reasons, then print it.

In [ ]:
REASONS = {
    "time": "",     # why this time base?
    "spike": "",    # why this rule, and what would confirm the flag?
    "frozen": "",   # why this run length? what false alarms could it raise?
    "fill": "",     # why this fill, for this question?
}
LIMITS = "Filled hours borrow other days' weather; night solar still fails the 5% check (see step 3)."
RERUN = "This notebook with MINE as set above."

def data_note(ch, reasons, limits, rerun):
    R, rep = clean(ch), report(clean(ch))
    c = R["counts"]
    lines = ["DATA NOTE",
             "Dataset: " + meta["title"],
             "Source: " + meta["source"] + " (" + meta["source_url"] + ")",
             "Files and SHA-256: " + "; ".join(f["name"] + " " + f["sha256"] for f in meta["files"]),
             "Accessed: " + meta["accessed"],
             "Extraction: " + meta["not_cleaned"],
             "Licence: " + meta["license"],
             "Time: " + meta["time"],
             "Cleaning log:",
             f"  1. Time base: {ch['time']}. Why: {reasons['time']}",
             f"  2. Spike rule: {ch['spike']}; {c['demand']['spike']} demand hour(s) flagged. Why: {reasons['spike']}",
             f"  3. Frozen-reading rule: {ch['frozen']} h; demand {c['demand']['frozen']} h, wind {c['wind']['frozen']} h, solar {c['solar']['frozen']} h flagged. Why: {reasons['frozen']}",
             f"  4. Fill: {ch['fill']}. Why: {reasons['fill']}",
             "  5. Effect: " + "; ".join(f"{k} {v}" for k, v in show(rep).items()) + " (raw: " + "; ".join(f"{k} {v}" for k, v in show(report(clean(RAW))).items()) + ")",
             "  6. Night check: %d night hours with solar above 5%% of its peak, %d of them filled" % night_solar(R),
             "Known limits: " + limits,
             "Rerun: " + rerun]
    return "\n".join(lines)

print(data_note(MINE, REASONS, LIMITS, RERUN))

## Part 3 · A spatial layer (QGIS, about 30 minutes)

The page's county is illustrative; here you use a real one: **Ford County, Kansas**, and the turbine locations of the
U.S. Wind Turbine Database (USGS, public domain; the USGS asks for an acknowledgment line). The free desktop GIS
[QGIS](https://qgis.org/) does every step.

1. Download the database as a CSV from its page (energy.usgs.gov/uswtdb) and note the version and the date.
2. In QGIS: *Layer → Add Layer → Add Delimited Text Layer*, with `xlong` and `ylat` as the point coordinates in
   EPSG:4326 (latitude and longitude in degrees).
3. Keep the county's turbines: *Filter* with `"t_state" = 'KS' AND "t_county" = 'Ford County'`. Count them.
4. For the 1 km buffers, reproject to the local UTM zone, **EPSG:32614** (UTM zone 14N, metres): *Vector → Data
   Management Tools → Reproject Layer*. Then *Vector → Geoprocessing Tools → Buffer* with 1,000 m, dissolved.
5. For the area, reproject the dissolved buffer to the equal-area **EPSG:5070** and compute `$area / 1e6` (km²) with
   the field calculator. Compute it in EPSG:32614 and in Web Mercator (EPSG:3857) too, and compare the three.
6. Log each step below: the file, the filter, each coordinate system, each output and its numbers.

In [ ]:
SPATIAL_LOG = [
    "Input: U.S. Wind Turbine Database, version ..., downloaded ... (public domain; acknowledgment: ...)",
    "Filter: t_state = 'KS' and t_county = 'Ford County': ... turbines",
    "Buffers: reprojected to EPSG:32614 (UTM 14N); 1,000 m buffers, dissolved",
    "Area: reprojected to EPSG:5070; ... km2 (EPSG:32614: ... km2; EPSG:3857: ... km2)",
]
print("\n".join(SPATIAL_LOG))

## Your turn (practice, about 2 hours in all)

1. **Your numbers (20 min).** Set `MINE` to your page choices; check the four numbers against the page.
2. **Sensitivity and the night check (30 min).** Read the table for your choices. Then add a night rule (B2's: in each
   month, an hour whose median solar is zero is night, and so are 22:00 to 05:00) after the fill, rerun the night
   check, and log the rule and its count.
3. **The spatial layer (30 min).** Part 3, every step logged.
4. **Licences (20 min).** For every source you use, write its licence and the attribution line it asks for.
5. **The note (20 min).** Fill `REASONS`, `LIMITS` and `RERUN`, print the note, and ask someone to rerun your numbers
   from it alone. The verification log and a validation against an independent source come in studio R5.

---
**Sources:** EIA, Hourly Electric Grid Monitor (Form EIA-930), six-month BALANCE files for 2025, balancing authority
SWPP, accessed 2 November 2026; USGS, U.S. Wind Turbine Database (licence and attribution on its page).

Text CC BY 4.0, code MIT: teachers and students may reuse and adapt it, with credit to Honglin Li. Corrections:
hlli@ksu.edu.